# Price model: cities A and B
Group assignment 2, deliverable 3.4. Target: `price` (nightly price in USD). Cities: **A = Los Angeles**, **B = San Francisco**. City C is not used anywhere here.

**Needs:** the Silver table `workspace.airbnb.silver_listings` of the 3.3 pipeline (both cities) and `src/price_model.py`. The notebook uses just the two listed cities.

**What lives where.** All reusable logic is in `src/price_model.py` (feature contract, split, baseline, model, metrics, scoring, saving). This notebook only calls it and shows the results, so the first run on city C is the same function calls on another table.

**Order of work (nothing is looked at on the test set before step 7).**
1. load listings and add features (no price information is used for features);
2. split train / test per city, **before any model choice**; the split is saved;
3. fit the baseline on the training rows;
4. choose the model with cross-validation **on the training rows only**;
5. evaluate once on the test rows: MAE in USD for A, B and both, against the baseline;
6. explain and inspect the errors;
7. write predictions for **every** listing to a gold table (dashboard and Genie read it) and save the model.

Every number reported in sections 6 to 8 comes from the held-out test rows.

## 1. Setup

In [0]:
import datetime
import importlib
import os
import shutil
import sys

import numpy as np
import pandas as pd
from pyspark.sql import functions as F

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

SRC_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
assert os.path.exists(os.path.join(SRC_DIR, "price_model.py")), f"price_model.py not found in {SRC_DIR}"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
import price_model as pm
importlib.reload(pm)

CATALOG, SCHEMA = "workspace", "airbnb_eda"        # model-evaluation objects stay here, separate from the dashboard's workspace.airbnb tables
SOURCE_TABLE = "workspace.airbnb.silver_listings"   # output of the 3.3 pipeline: one row per listing, both cities
COLUMN_MAP = {}                                     # Silver column name -> model column name, only if the names differ from pm.REQUIRED_COLUMNS
EXPECTED_ROWS = {"Los_Angeles": 43781, "San_Francisco": 7405}   # listings per city in the Silver table (3.3 test run)
ESSENTIAL = ["city", "id", "room_type", "price", "accommodates", "latitude", "longitude"]
DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "data"))      # data/listings_<City>.csv.gz in the repo
CITY_ROLE = {"Los_Angeles": "A", "San_Francisco": "B"}        # city C is intentionally not listed
COLORS = {"Los_Angeles": "#1f77b4", "San_Francisco": "#ff7f0e"}
TEST_SIZE, SEED = 0.2, pm.SEED
MODEL_NAME = "price_model_v1"                      # next version: price_model_v2, its rows are written next to v1
REFIT = False                                      # False: reuse the stored split and the saved v1 model when they exist; True: split and fit again
DASHBOARD_OWNER = ""                               # user (email) or group that gets read access to the predictions, see section 10
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.raw")      # the model file is saved here
MODEL_DIR = f"/Volumes/{CATALOG}/{SCHEMA}/raw/models"
RESULTS_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "results"))   # small result files that go into the repo
GOLD = {"predictions": "workspace.airbnb.gold_predictions",                  # published for the dashboard
        "metrics": f"{CATALOG}.{SCHEMA}.gold_price_model_metrics",           # model evaluation, NOT the dashboard's workspace.airbnb.gold_model_metrics
        "split": f"{CATALOG}.{SCHEMA}.price_model_split"}


def table_exists(name):
    return spark.catalog.tableExists(name)


def read_table(name):
    return spark.table(name)

## 2. Load listings and add features

The listings come from the Silver table of the 3.3 pipeline. `load_source` takes the columns listed in `pm.REQUIRED_COLUMNS` (renamed with `COLUMN_MAP` if the Silver names differ) and `pm.add_features` adds the model features. A missing column is filled with missing values and a warning is printed.

In [0]:
def load_source(cities):
    src = spark.table(SOURCE_TABLE)
    src = src.select([F.col(c).alias(COLUMN_MAP.get(c, c)) for c in src.columns])
    print("cities in", SOURCE_TABLE, ":", sorted(r[0] for r in src.select("city").distinct().collect()))
    have = [c for c in pm.REQUIRED_COLUMNS + pm.OPTIONAL_COLUMNS if c in src.columns]
    return pm.add_features(src.filter(F.col("city").isin(list(cities))).select(have).toPandas()).reset_index(drop=True)


print("required columns:", pm.REQUIRED_COLUMNS)
data = load_source(list(CITY_ROLE))
data["price_status"] = pm.price_status(data)
print(data.shape)

required columns: ['city', 'id', 'room_type', 'property_type', 'price', 'accommodates', 'bedrooms', 'beds', 'bathrooms_num', 'bathroom_shared', 'n_amenities', 'minimum_nights', 'host_is_superhost', 'host_listings_count', 'calculated_host_listings_count', 'hosts_time_as_host_years', 'has_license', 'latitude', 'longitude']
cities in workspace.airbnb.silver_listings : ['Los_Angeles', 'San_Francisco']
(51186, 29)


**Source check.** One row per required column: % missing in the source. The run stops if the listings per city differ from the 3.3 test run, or if an essential column is missing or empty (then check the Silver column names and `COLUMN_MAP`).

In [0]:
cnt = data.groupby("city").size().to_dict()
assert cnt == EXPECTED_ROWS, f"listings per city in the source: {cnt}, expected {EXPECTED_ROWS}"
quality = (100 * data[pm.REQUIRED_COLUMNS].isna().mean()).round(2).rename("pct_missing").rename_axis("column").reset_index()
display(quality)
empty = quality.loc[quality.pct_missing == 100, "column"].tolist()
bad = sorted(set(ESSENTIAL) & set(empty))
assert not bad, f"essential columns are missing or empty in the source: {bad}. Check the Silver column names and COLUMN_MAP."
if empty:
    print("WARNING: columns that are completely empty in the source (the model sees no information in them):", empty)

column,pct_missing
city,0.0
id,0.0
room_type,0.0
property_type,0.0
price,17.21
accommodates,0.0
bedrooms,0.18
beds,2.69
bathrooms_num,0.18
bathroom_shared,0.18


**Listings by price status.** One row per city: all listings, listings with a valid price (used for fitting and testing), without any price, and placeholder prices (excluded from fitting and scoring).

In [0]:
status = data.groupby(["city", "price_status"]).size().unstack("price_status").fillna(0).astype(int)
status.insert(0, "all_listings", status.sum(axis=1))
display(status.rename_axis("city").reset_index())

city,all_listings,no_price,placeholder,valid
Los_Angeles,43781,7094,11,36676
San_Francisco,7405,1714,15,5676


## 3. Split: train and test, before any model choice

Listings with a valid price are split 80 / 20, stratified by city and room type, with a fixed seed. Listings without a valid price are marked `not_labelled`: they get predictions later but are never used for fitting or scoring. The split is saved so everybody uses the same rows; a stored split is reused as it is (`REFIT = False`), so the test rows never change.

In [0]:
split_reused = table_exists(GOLD["split"]) and not REFIT
if split_reused:
    stored = read_table(GOLD["split"]).toPandas()[["city", "id", "split"]].rename(columns={"split": "stored_split"})
    stored["id"] = stored["id"].astype("int64")
    data = data.merge(stored, on=["city", "id"], how="left")
    valid = pm.valid_price_mask(data)
    in_split = data["stored_split"].isin(["train", "test"])
    data["split"] = np.where(valid & in_split, data["stored_split"], "not_labelled")
    print(f"stored split reused from {GOLD['split']}; listings with a valid price but no stored label (left not_labelled): {int((valid & ~in_split).sum())}")
    data = data.drop(columns="stored_split")
else:
    data["split"] = pm.make_split(data, test_size=TEST_SIZE, seed=SEED)

train = data[data.split == "train"].reset_index(drop=True)
test = data[data.split == "test"].reset_index(drop=True)
overlap = set(zip(train.city, train.id)) & set(zip(test.city, test.id))
assert not overlap, "a listing is in both train and test"

if split_reused:
    print("split table unchanged")
else:
    spark.createDataFrame(data[["city", "id", "split"]].astype({"id": "int64"})).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(GOLD["split"])
    print(f"new split saved to {GOLD['split']}")

stored split reused from workspace.airbnb_eda.price_model_split; listings with a valid price but no stored label (left not_labelled): 0
split table unchanged


**Split sizes.** One row per city, one column per split, plus the share of valid-price listings in the test set.

In [0]:
sizes = data.groupby(["city", "split"]).size().unstack("split").fillna(0).astype(int)
sizes["test_share_of_labelled_pct"] = (100 * sizes["test"] / (sizes["train"] + sizes["test"])).round(1)
display(sizes.rename_axis("city").reset_index())

city,not_labelled,test,train,test_share_of_labelled_pct
Los_Angeles,7105,7335,29341,20.0
San_Francisco,1729,1136,4540,20.0


**Do train and test look alike?** One row per (city, split): number of listings, quartiles of price, share of entire homes and share of monthly rentals. They should be close.

In [0]:
chk = (data[data.split.isin(["train", "test"])].assign(entire_home=lambda x: (x.room_type == "Entire home/apt").astype(float))
       .groupby(["city", "split"]).agg(listings=("id", "size"), price_q1=("price", lambda s: s.quantile(.25)),
                                       price_median=("price", "median"), price_q3=("price", lambda s: s.quantile(.75)),
                                       pct_entire_home=("entire_home", lambda s: 100 * s.mean()),
                                       pct_monthly_rental=("is_monthly_rental", lambda s: 100 * s.mean())).round(1))
display(chk.reset_index())

city,split,listings,price_q1,price_median,price_q3,pct_entire_home,pct_monthly_rental
Los_Angeles,test,7335,114.8,213.1,365.0,74.1,45.0
Los_Angeles,train,29341,116.2,215.0,379.0,74.1,44.7
San_Francisco,test,1136,156.2,259.0,482.7,56.8,36.1
San_Francisco,train,4540,159.5,268.0,477.9,56.8,35.1


## 4. Baseline

The baseline predicts, for every listing, **the median training price of listings with the same room type and number of guests** (fallback for a combination never seen in training: the room-type median, then the overall median). A second baseline also splits by stay type (short / monthly), because the EDA found that monthly listings cost about 40 % less.

In [0]:
saved_path = f"{MODEL_DIR}/{MODEL_NAME}.joblib"
USE_SAVED = os.path.exists(saved_path) and not REFIT
if USE_SAVED:
    saved = pm.load_model(saved_path)
    baseline, baseline_stay = saved["baseline"], saved["baseline_stay_type"]
    print("reusing the baselines and the fitted model saved in", saved_path)
else:
    baseline = pm.MedianBaseline(("room_type", "accommodates")).fit(train)
    baseline_stay = pm.MedianBaseline(("room_type", "accommodates", "stay_type")).fit(train)
print("baseline cells:", len(baseline.table_), "| with stay type:", len(baseline_stay.table_), "| overall median price:", round(baseline.global_, 2))

reusing the baselines and the fitted model saved in /Volumes/workspace/airbnb_eda/raw/models/price_model_v1.joblib
baseline cells: 51 | with stay type: 99 | overall median price: 222.0


**Baseline lookup table** (training data only). One row per (room type, guests): median price and number of training listings behind it. The 15 most common combinations.

In [0]:
display(baseline.table_.reset_index().sort_values("n_listings", ascending=False).head(15).round(2))

room_type,accommodates,median_price,n_listings
Entire home/apt,2.0,162.85,6050
Entire home/apt,4.0,245.0,5272
Private room,2.0,101.0,5171
Entire home/apt,6.0,365.78,3621
Private room,1.0,71.5,2264
Entire home/apt,8.0,487.0,2147
Entire home/apt,3.0,198.0,2086
Entire home/apt,5.0,299.78,1548
Entire home/apt,10.0,714.0,1043
Private room,4.0,235.0,737


## 5. Features

**Feature list.** One row per feature: its type, meaning, and why it is allowed. The assertion below fails if any column from the EDA leakage review (prices and price-derived columns, quote details, availability, reviews, identifiers) is among the features.

In [0]:
pm.assert_no_leakage()
print("no leaking column among the", len(pm.FEATURES), "features")
display(pd.DataFrame(pm.FEATURE_DOC, columns=["feature", "type", "meaning", "why it is allowed"]))

no leaking column among the 20 features


feature,type,meaning,why it is allowed
room_type,categorical,"Entire home, private room, hotel room or shared room","Property attribute; overlaps with property_type, so its own permutation importance is small"
property_type,categorical,"Type of property (the 25 most common in training, the rest grouped as Other)",Property attribute; grouping avoids a long tail of rare levels
accommodates,number,Number of guests,Property attribute; drives price
bedrooms,number,Bedrooms (0 = studio),Property attribute
beds,number,Beds,Property attribute
bathrooms_num,number,Bathrooms parsed from the text (half-bath = 0.5),Property attribute
bathroom_shared,0/1,Bathroom is shared,Property attribute
n_amenities,number,Number of amenities listed,Property attribute; robust summary of a free-text list
minimum_nights_capped,number,"Host's minimum stay, capped at 365",Booking rule set by the host; carries the quoted stay length
is_monthly_rental,0/1,Minimum stay of 28 nights or more,Monthly listings are priced about 40 % lower per night


**Missing values in the features** (training rows). One row per feature: % missing. Boosting handles them natively; ridge imputes the median and adds a missing-indicator.

In [0]:
miss = (100 * train[pm.FEATURES].isna().mean()).round(2).rename("pct_missing").rename_axis("feature").reset_index()
display(miss.sort_values("pct_missing", ascending=False))

feature,pct_missing
beds,2.86
bedrooms,0.21
bathrooms_num,0.2
bathroom_shared,0.2
minimum_nights_capped,0.08
is_monthly_rental,0.08
room_type,0.0
hosts_time_as_host_years,0.0
dx_km,0.0
dist_to_center_km,0.0


## 6. Choosing the model (training rows only)

Five candidates (two ridge regressions, three boosting settings) and the baseline are compared with 5-fold cross-validation **inside the training rows**. The test rows are not touched. Metric: MAE in USD per night on the held-out fold; `cv_mae_std` = spread over the 5 folds.

In [0]:
if USE_SAVED:
    best_name = saved["meta"]["chosen_candidate"]
    best_kind, best_params = pm.CANDIDATES[best_name]
    print("cross-validation skipped, the saved model is reused; chosen when it was fitted:", best_name, best_params)
else:
    rows = []
    cv = pm.cross_validate(lambda: pm.MedianBaseline(("room_type", "accommodates")), train)
    rows.append({"candidate": "baseline: room type x guests", "kind": "baseline", **cv})
    for name, (kind, params) in pm.CANDIDATES.items():
        cv = pm.cross_validate(lambda kind=kind, params=params: pm.PriceModel(kind, params), train)
        rows.append({"candidate": name, "kind": kind, **cv})
    selection = pd.DataFrame(rows).sort_values("cv_mae_mean").reset_index(drop=True)
    display(selection.round(2))

    best_name = selection[selection.kind != "baseline"].iloc[0]["candidate"]
    best_kind, best_params = pm.CANDIDATES[best_name]
    print("chosen:", best_name, best_params)

cross-validation skipped, the saved model is reused; chosen when it was fitted: boosting, 31 leaves {'max_depth': None, 'max_leaf_nodes': 31, 'learning_rate': 0.05, 'max_iter': 500, 'min_samples_leaf': 20, 'l2_regularization': 1.0}


**Final fit.** The chosen candidate is fitted once on all training rows (or the saved v1 model is reused).

In [0]:
model = saved["model"] if USE_SAVED else pm.PriceModel(best_kind, best_params).fit(train)
print(model.describe()["kind"], "| training listings:", model.n_train_, "| property types kept:", len(model.top_types_), "+ Other")

hgb | training listings: 33881 | property types kept: 25 + Other


## 7. Evaluation on the test rows

Predictions for **all** listings are produced once (train, test and unlabelled), in the layout of the gold table. Metrics below use the test rows only.

In [0]:
scored = pm.score_listings(data, model, baseline, baseline_stay, split=data["split"], model_name=MODEL_NAME,
                           scored_at=f"{datetime.datetime.now():%Y-%m-%d %H:%M:%S}")
test_scored = scored[scored.split == "test"].reset_index(drop=True)
PRED = {"baseline": "baseline_price", "baseline + stay type": "baseline_stay_type_price", "model": "predicted_price"}
scopes = pm.city_scopes(test_scored, CITY_ROLE)
results = pm.metrics_table(test_scored, PRED, scopes)
print("test listings:", len(test_scored))

test listings: 8471


**Test-set error.** One row per (scope, model). `mae` = mean absolute error in USD per night (the assignment's metric); `median_ae` = typical error; `rmse` = punishes large errors; `mape_pct` = mean absolute % error; `mean_error` = predicted minus actual (negative = under-prices on average); `r2_log` = R-squared on log price.

In [0]:
display(results.round(2))

scope,model,n,mae,median_ae,rmse,mape_pct,mean_error,r2_log
A: Los_Angeles,baseline,7335,180.49,67.86,579.77,48.43,-90.01,0.55
A: Los_Angeles,baseline + stay type,7335,171.6,57.0,578.68,41.36,-88.59,0.61
A: Los_Angeles,model,7335,110.83,36.66,399.65,25.54,-34.56,0.84
B: San_Francisco,baseline,1136,245.8,84.89,556.05,43.65,-204.25,0.07
B: San_Francisco,baseline + stay type,1136,227.33,69.5,540.04,36.54,-192.45,0.3
B: San_Francisco,model,1136,139.26,53.48,355.97,29.3,-53.38,0.78
A+B: both,baseline,8471,189.25,69.75,576.64,47.79,-105.33,0.49
A+B: both,baseline + stay type,8471,179.08,58.98,573.65,40.72,-102.52,0.58
A+B: both,model,8471,114.65,38.74,394.07,26.05,-37.09,0.84


**Improvement over the baseline.** One row per scope: MAE of each, and how much lower the model's MAE is than the baseline's (positive = better).

In [0]:
wide = results.pivot(index="scope", columns="model", values="mae")[["baseline", "baseline + stay type", "model"]].reindex(list(scopes))
wide["model_vs_baseline_pct"] = (100 * (1 - wide["model"] / wide["baseline"])).round(1)
wide["model_vs_baseline_stay_pct"] = (100 * (1 - wide["model"] / wide["baseline + stay type"])).round(1)
display(wide.round(2).rename_axis("scope").reset_index())


scope,baseline,baseline + stay type,model,model_vs_baseline_pct,model_vs_baseline_stay_pct
A: Los_Angeles,180.49,171.6,110.83,38.6,35.4
B: San_Francisco,245.8,227.33,139.26,43.3,38.7
A+B: both,189.25,179.08,114.65,39.4,36.0


**Error by stay type.** One row per (city, stay type): test listings and MAE of each model.

In [0]:
display(pm.mae_by(test_scored, ["city", "stay_type"], PRED).round(2))

city,stay_type,n,mae_baseline,mae_baseline + stay type,mae_model
Los_Angeles,monthly,3299,152.03,136.62,91.68
Los_Angeles,short,4033,203.86,200.28,126.55
Los_Angeles,unknown,3,54.84,84.37,49.31
San_Francisco,monthly,410,75.19,77.39,55.8
San_Francisco,short,725,342.5,312.35,186.51
San_Francisco,unknown,1,83.15,62.5,101.42


**Error by room type.** One row per (city, room type).

In [0]:
display(pm.mae_by(test_scored, ["city", "room_type"], PRED).round(2))

city,room_type,n,mae_baseline,mae_baseline + stay type,mae_model
Los_Angeles,Entire home/apt,5434,214.88,204.42,132.85
Los_Angeles,Hotel room,79,205.94,206.53,108.07
Los_Angeles,Private room,1765,77.25,72.74,45.27
Los_Angeles,Shared room,57,63.47,56.07,45.46
San_Francisco,Entire home/apt,645,263.53,249.61,149.23
San_Francisco,Hotel room,31,423.62,419.73,169.15
San_Francisco,Private room,455,209.81,183.78,124.39
San_Francisco,Shared room,5,130.59,122.25,19.46


**Error by number of guests.** One row per (city, guest group).

In [0]:
test_scored["guests"] = pd.cut(test_scored["accommodates"], [0, 1, 2, 4, 6, 8, 100], labels=["1", "2", "3-4", "5-6", "7-8", "9+"])
display(pm.mae_by(test_scored, ["city", "guests"], PRED).round(2))

city,guests,n,mae_baseline,mae_baseline + stay type,mae_model
Los_Angeles,1,620,39.64,37.66,28.38
Los_Angeles,2,2446,78.74,67.54,44.73
Los_Angeles,3-4,1781,126.24,113.47,80.14
Los_Angeles,5-6,1288,195.91,189.99,125.38
Los_Angeles,7-8,632,316.78,312.17,203.89
Los_Angeles,9+,568,755.85,750.14,445.19
San_Francisco,1,117,59.6,52.5,34.33
San_Francisco,2,452,196.0,177.67,127.54
San_Francisco,3-4,346,195.59,175.74,114.0
San_Francisco,5-6,147,448.16,427.41,224.43


## 8. Error analysis (test rows)

**Predicted vs actual price, by price band.** One row per (city, band of the actual price): test listings, median actual and median predicted price, median of predicted / actual (1 = exact; below 1 = under-priced) and the model's MAE.

In [0]:
d = test_scored[test_scored.error_usd.notna()].copy()
d["actual_price_band"] = pd.cut(d["actual_price"], [0, 100, 200, 400, 800, 1600, 1e9],
                                labels=["under $100", "$100-200", "$200-400", "$400-800", "$800-1,600", "$1,600 and more"])
d["ratio_pred_to_actual"] = d["predicted_price"] / d["actual_price"]
display(d.groupby(["city", "actual_price_band"], observed=True).agg(
    listings=("id", "size"), median_actual=("actual_price", "median"), median_predicted=("predicted_price", "median"),
    median_ratio=("ratio_pred_to_actual", "median"), mae=("abs_error_usd", "mean")).round(2).reset_index())


city,actual_price_band,listings,median_actual,median_predicted,median_ratio,mae
Los_Angeles,under $100,1473,69.43,76.68,1.11,16.99
Los_Angeles,$100-200,1992,144.3,153.85,1.09,35.76
Los_Angeles,$200-400,2296,279.86,282.0,1.02,66.06
Los_Angeles,$400-800,1030,528.12,463.65,0.87,149.49
Los_Angeles,"$800-1,600",366,1035.89,839.54,0.8,358.84
Los_Angeles,"$1,600 and more",178,2329.78,1744.14,0.64,1571.41
San_Francisco,under $100,116,74.72,81.51,1.11,20.36
San_Francisco,$100-200,309,154.07,178.48,1.19,49.02
San_Francisco,$200-400,368,279.39,291.87,1.04,72.75
San_Francisco,$400-800,212,549.38,461.94,0.86,157.92


**Size of the errors.** One row per city: quantiles of log(predicted / actual) (0 = exact, +0.69 = predicted twice the actual, -0.69 = half) and of the absolute error in USD (p50 = typical, p90 and p99 = bad cases).

In [0]:
d = test_scored[test_scored.error_usd.notna()].copy()
d["log_ratio"] = np.log(d["predicted_price"] / d["actual_price"])
rows = []
for city, g in d.groupby("city"):
    r = {"city": city, "listings": len(g)}
    for p in (0.05, 0.25, 0.5, 0.75, 0.95):
        r[f"log_ratio_p{int(p * 100)}"] = g["log_ratio"].quantile(p)
    for p in (0.5, 0.9, 0.99):
        r[f"abs_error_usd_p{int(p * 100)}"] = g["abs_error_usd"].quantile(p)
    rows.append(r)
display(pd.DataFrame(rows).round(3))


city,listings,log_ratio_p5,log_ratio_p25,log_ratio_p50,log_ratio_p75,log_ratio_p95,abs_error_usd_p50,abs_error_usd_p90,abs_error_usd_p99
Los_Angeles,7335,-0.574,-0.161,0.032,0.209,0.501,36.66,205.022,1377.98
San_Francisco,1136,-0.672,-0.212,0.031,0.226,0.556,53.48,305.565,1510.027


**Calibration.** Test listings are cut into 10 groups by predicted price; for each group the average predicted and the average actual price. Close columns mean the model is right on average at every price level.

In [0]:
d = test_scored[test_scored.error_usd.notna()].copy()
d["predicted_decile"] = pd.qcut(d.predicted_price, 10, labels=False, duplicates="drop") + 1
cal = d.groupby("predicted_decile").agg(listings=("id", "size"), mean_predicted=("predicted_price", "mean"),
                                        mean_actual=("actual_price", "mean"), mean_abs_error=("abs_error_usd", "mean")).round(1)
display(cal.reset_index())

predicted_decile,listings,mean_predicted,mean_actual,mean_abs_error
1,848,57.5,61.3,12.6
2,847,96.3,99.4,23.2
3,847,128.2,140.7,38.3
4,847,163.5,180.3,53.6
5,847,203.8,217.9,62.2
6,847,250.0,265.7,68.1
7,848,300.3,331.3,90.7
8,846,371.4,401.6,117.3
9,847,491.5,543.4,166.1
10,847,1192.5,1384.4,514.7


**Error by neighbourhood.** One row per (city, neighbourhood) with at least 15 test listings: MAE and mean error (negative = under-priced by the model). The 8 neighbourhoods with the largest MAE per city.

In [0]:
for city in CITY_ROLE:
    g = (test_scored[(test_scored.city == city) & test_scored.error_usd.notna()].groupby("neighbourhood_cleansed")
         .agg(test_listings=("id", "size"), median_actual_price=("actual_price", "median"), mae=("abs_error_usd", "mean"),
              mean_error=("error_usd", "mean")).query("test_listings >= 15").sort_values("mae", ascending=False).head(8).round(1))
    print(f"{CITY_ROLE[city]}: {city}")
    display(g.reset_index())

A: Los_Angeles


neighbourhood_cleansed,test_listings,median_actual_price,mae,mean_error
Bel-Air,17,2040.7,1904.1,-475.1
Malibu,76,798.0,874.0,-584.7
Beverly Crest,28,1685.8,622.5,51.9
Unincorporated Santa Monica Mountains,33,691.6,551.1,-205.9
Hollywood Hills West,97,1012.0,529.6,-186.6
Beverly Hills,115,322.4,314.8,-215.5
Brentwood,31,182.7,312.0,-222.8
Mount Washington,16,184.0,266.3,-217.6


B: San_Francisco


neighbourhood_cleansed,test_listings,median_actual_price,mae,mean_error
Nob Hill,47,228.2,244.6,-90.2
Russian Hill,24,439.1,233.6,-85.9
Western Addition,98,212.5,224.4,-106.0
Pacific Heights,21,306.0,222.1,-104.8
South of Market,92,277.4,204.8,-131.1
Financial District,44,361.6,204.5,-76.6
North Beach,51,760.6,181.1,-63.6
Chinatown,17,230.0,180.5,-127.0


**Listings the model under-prices most** (predicted far below actual): the 12 test listings with the most negative error.

In [0]:
COLS = ["city", "id", "room_type", "property_type", "accommodates", "bedrooms", "minimum_nights", "actual_price", "predicted_price", "error_usd", "pct_error"]
display(test_scored.nsmallest(12, "error_usd")[COLS])

city,id,room_type,property_type,accommodates,bedrooms,minimum_nights,actual_price,predicted_price,error_usd,pct_error
Los_Angeles,1079908245408870997,Entire home/apt,Entire home,14.0,6.0,4.0,18117.0,5265.72,-12851.28,-70.93
Los_Angeles,877775930689549848,Entire home/apt,Entire home,16.0,4.0,1.0,11412.0,1816.33,-9595.67,-84.08
Los_Angeles,1136418613533078238,Entire home/apt,Entire home,16.0,9.0,31.0,14553.21,5133.9,-9419.31,-64.72
Los_Angeles,1660092723965499150,Entire home/apt,Entire rental unit,16.0,0.0,3.0,10000.0,667.13,-9332.87,-93.33
Los_Angeles,1772607552574277623,Entire home/apt,Entire rental unit,8.0,3.0,31.0,9225.45,319.54,-8905.91,-96.54
Los_Angeles,21770892,Entire home/apt,Entire home,8.0,4.0,30.0,8314.77,1646.07,-6668.7,-80.2
San_Francisco,1668906000013864396,Entire home/apt,Entire home,8.0,4.0,1.0,7590.0,1295.04,-6294.96,-82.94
Los_Angeles,51126372,Entire home/apt,Entire villa,10.0,6.0,3.0,10713.67,4799.53,-5914.14,-55.2
Los_Angeles,1648547808532129741,Entire home/apt,Entire home,16.0,9.0,30.0,10113.13,5682.69,-4430.44,-43.81
Los_Angeles,1544331495606343190,Entire home/apt,Entire home,16.0,9.0,30.0,5639.37,1253.01,-4386.36,-77.78


**Listings the model over-prices most** (predicted far above actual): the 12 test listings with the largest positive error.

In [0]:
display(test_scored.nlargest(12, "error_usd")[COLS])

city,id,room_type,property_type,accommodates,bedrooms,minimum_nights,actual_price,predicted_price,error_usd,pct_error
Los_Angeles,899099244009918926,Entire home/apt,Entire villa,16.0,8.0,2.0,3112.0,6897.53,3785.53,121.64
Los_Angeles,47903680,Entire home/apt,Entire home,16.0,8.0,2.0,1992.5,4263.21,2270.71,113.96
Los_Angeles,38902329,Entire home/apt,Entire villa,16.0,9.0,1.0,3374.0,5641.27,2267.27,67.2
Los_Angeles,1461320055876641100,Entire home/apt,Entire home,10.0,6.0,30.0,1831.0,4070.88,2239.88,122.33
San_Francisco,1143919288844859087,Entire home/apt,Entire rental unit,2.0,1.0,1.0,490.4,2708.04,2217.64,452.21
Los_Angeles,31309299,Entire home/apt,Entire home,7.0,6.0,1.0,269.23,2029.74,1760.51,653.91
Los_Angeles,1227696600344472592,Entire home/apt,Entire home,16.0,7.0,2.0,1364.34,2935.18,1570.84,115.14
Los_Angeles,53891249,Entire home/apt,Entire villa,16.0,6.0,30.0,1802.26,3356.31,1554.05,86.23
San_Francisco,1144753179973491172,Entire home/apt,Entire rental unit,2.0,1.0,1.0,365.6,1887.66,1522.06,416.32
Los_Angeles,1764298957404027609,Entire home/apt,Entire home,10.0,5.0,3.0,2579.95,4092.58,1512.63,58.63


## 9. What the model uses

**Permutation importance** (test rows). Each group of features is shuffled across listings and the model's MAE is recomputed: `mae_increase_usd` = how many dollars worse the predictions get without that information (5 repeats; `mae_increase_std` = spread). Location features are shuffled together because they overlap.

In [0]:
eval_rows = data[(data.split == "test")].reset_index(drop=True)
importance = pm.permutation_importance_mae(model, eval_rows, n_repeats=5, seed=SEED)
importance["bar"] = ["#" * int(round(30 * v / importance["mae_increase_usd"].max())) for v in importance["mae_increase_usd"]]
display(importance.round(2))


feature_group,features,mae_increase_usd,mae_increase_std,mae_increase_pct,bar
size and capacity,"accommodates, bedrooms, beds, bathrooms_num, bathroom_shared",116.69,1.46,101.79,##############################
location,"latitude, longitude, dist_to_center_km, dx_km, dy_km",56.92,0.56,49.65,###############
stay rules,"minimum_nights_capped, is_monthly_rental",38.91,0.75,33.94,##########
property type,property_type,22.46,0.84,19.59,######
host,"host_is_superhost, log_host_listings_count, log_calc_host_listings_count, hosts_time_as_host_years, has_license",13.51,0.37,11.78,###
amenities,n_amenities,6.51,0.32,5.67,##
room type,room_type,0.46,0.08,0.4,


## 10. Predictions for every listing (gold table), result files and saved model

`gold_predictions` has one row per listing (train, test and not labelled): actual and predicted price, both baselines, errors, the split and the `model_name`. **A run replaces only its own rows** (same `model_name`, same cities), so a later `price_model_v2` run or the run on city C adds rows next to these instead of overwriting them, and the dashboard can compare versions. Column comments are set so Genie understands the columns. `gold_price_model_metrics` holds the test metrics the same way.

In [0]:
def sql_list(values):
    return ", ".join("'" + str(v).replace("'", "''") + "'" for v in sorted(set(values)))

def write_rows(sdf, table, replace_where):
    # replace only the rows that match replace_where; the first run creates the table and returns True.
    # An existing table with different columns is left alone (it may belong to someone else).
    if table_exists(table):
        existing = set(read_table(table).columns)
        if existing != set(sdf.columns):
            raise ValueError(f"{table} exists with different columns, not touching it.\nexisting: {sorted(existing)}\nnew: {sorted(sdf.columns)}")
        sdf.write.mode("overwrite").option("replaceWhere", replace_where).saveAsTable(table)
        return False
    sdf.write.saveAsTable(table)
    return True

gold = scored.astype(object).where(scored.notna(), None)
created = write_rows(spark.createDataFrame(gold, schema=pm.GOLD_DDL), GOLD["predictions"],
                     f"model_name = '{MODEL_NAME}' AND city IN ({sql_list(scored.city)})")
if created:
    for col, comment in pm.GOLD_COMMENTS.items():
        spark.sql(f"ALTER TABLE {GOLD['predictions']} ALTER COLUMN {col} COMMENT '{comment}'")
    spark.sql(f"COMMENT ON TABLE {GOLD['predictions']} IS 'Model price prediction for every listing with actual price, baselines and errors; evaluate on rows where split = test'")

metrics_out = results.assign(split="test", model_name=MODEL_NAME, evaluated_at=f"{datetime.datetime.now():%Y-%m-%d %H:%M:%S}")
write_rows(spark.createDataFrame(metrics_out), GOLD["metrics"],
           f"model_name = '{MODEL_NAME}' AND scope IN ({sql_list(results.scope)})")
print("written:", GOLD["predictions"], "and", GOLD["metrics"])

written: workspace.airbnb.gold_predictions and workspace.airbnb_eda.gold_price_model_metrics


**Check.** One row per (city, split): listings of this model version in the gold table against listings in the source data. They must be equal.

In [0]:
in_gold = (read_table(GOLD["predictions"]).filter((F.col("model_name") == MODEL_NAME) & F.col("city").isin(list(CITY_ROLE)))
           .groupBy("city", "split").agg(F.count("*").alias("rows_in_gold_table")).toPandas())
expected = data.groupby(["city", "split"]).size().rename("rows_in_data").reset_index()
check = expected.merge(in_gold, on=["city", "split"], how="outer")
assert (check.rows_in_data == check.rows_in_gold_table).all(), "gold table does not contain every listing"
display(check)

city,split,rows_in_data,rows_in_gold_table
Los_Angeles,not_labelled,7105,7105
Los_Angeles,test,7335,7335
Los_Angeles,train,29341,29341
San_Francisco,not_labelled,1729,1729
San_Francisco,test,1136,1136
San_Francisco,train,4540,4540


**Save the model** (with both baselines and a JSON description) to the volume, so the run on city C can load it without refitting.

In [0]:
meta = {"model_name": MODEL_NAME, "trained_at": saved["meta"].get("trained_at", "unknown") if USE_SAVED else f"{datetime.datetime.now():%Y-%m-%d %H:%M:%S}", "source_table": SOURCE_TABLE,
        "cities": list(CITY_ROLE), "test_size": TEST_SIZE, "chosen_candidate": best_name,
        "test_mae": {f"{r.scope} | {r.model}": round(r.mae, 2) for r in results.itertuples()}}
if USE_SAVED:
    print("the saved model was reused, it is not saved again:", saved_path)
else:
    tmp_path = f"/tmp/{MODEL_NAME}.joblib"
    pm.save_model(tmp_path, model, baseline, baseline_stay, meta)
    os.makedirs(MODEL_DIR, exist_ok=True)
    for ext in (".joblib", ".json"):
        shutil.copyfile(f"/tmp/{MODEL_NAME}{ext}", f"{MODEL_DIR}/{MODEL_NAME}{ext}")
    print("saved to", f"{MODEL_DIR}/{MODEL_NAME}.joblib")

the saved model was reused, it is not saved again: /Volumes/workspace/airbnb_eda/raw/models/price_model_v1.joblib


**Results for the repo.** Notebook outputs are not kept by every export or commit, so the test-set table is also written as a small CSV and Markdown file in `results/`. Commit both and quote the table in the README.

In [0]:
os.makedirs(RESULTS_DIR, exist_ok=True)
results.round(3).to_csv(f"{RESULTS_DIR}/{MODEL_NAME}_test_metrics.csv", index=False)
with open(f"{RESULTS_DIR}/{MODEL_NAME}_test_metrics.md", "w") as f:
    f.write(pm.metrics_to_markdown(results, meta))
print("written to", RESULTS_DIR)
print(pm.metrics_to_markdown(results, meta))

written to /Workspace/Repos/bobas.pn@ucu.edu.ua/PizzaByte_group_assignment_2/results
Model: price_model_v1, chosen candidate: boosting, 31 leaves, trained at 2026-11-11 22:28:02

| scope | model | n | mae | median_ae | rmse | mape_pct | mean_error | r2_log |
|---|---|---|---|---|---|---|---|---|
| A: Los_Angeles | baseline | 7335 | 180.49 | 67.86 | 579.77 | 48.43 | -90.01 | 0.55 |
| A: Los_Angeles | baseline + stay type | 7335 | 171.60 | 57.00 | 578.68 | 41.36 | -88.59 | 0.61 |
| A: Los_Angeles | model | 7335 | 110.83 | 36.66 | 399.65 | 25.54 | -34.56 | 0.84 |
| B: San_Francisco | baseline | 1136 | 245.80 | 84.89 | 556.05 | 43.65 | -204.25 | 0.07 |
| B: San_Francisco | baseline + stay type | 1136 | 227.33 | 69.50 | 540.04 | 36.54 | -192.45 | 0.30 |
| B: San_Francisco | model | 1136 | 139.26 | 53.48 | 355.97 | 29.30 | -53.38 | 0.78 |
| A+B: both | baseline | 8471 | 189.25 | 69.75 | 576.64 | 47.79 | -105.33 | 0.49 |
| A+B: both | baseline + stay type | 8471 | 179.08 | 58.98 | 573.65 | 40

## 11. Summary and how to reuse

In [0]:
print(f"Model: {best_name} (fitted on log price, converted back to USD), {model.n_train_:,} training listings, {len(pm.FEATURES)} features")
print(f"Test set: {len(test_scored):,} listings, split {int((1 - TEST_SIZE) * 100)}/{int(TEST_SIZE * 100)} per city\n")
print(f"{'scope':<22}{'baseline MAE':>14}{'baseline+stay':>15}{'model MAE':>12}{'model vs baseline':>20}")
for scope in scopes:
    r = results[results.scope == scope].set_index("model")["mae"]
    print(f"{scope:<22}{r['baseline']:>13.2f}${r['baseline + stay type']:>14.2f}${r['model']:>11.2f}${100 * (1 - r['model'] / r['baseline']):>18.1f} %")

Model: boosting, 31 leaves (fitted on log price, converted back to USD), 33,881 training listings, 20 features
Test set: 8,471 listings, split 80/20 per city

scope                   baseline MAE  baseline+stay   model MAE   model vs baseline
A: Los_Angeles               180.49$        171.60$     110.83$              38.6 %
B: San_Francisco             245.80$        227.33$     139.26$              43.3 %
A+B: both                    189.25$        179.08$     114.65$              39.4 %


**Reuse for the next steps**
- **Dashboard and Genie:** read `workspace.airbnb.gold_predictions` (one row per listing and model version; filter `model_name` and use `split = 'test'` for honest error numbers). The model's own test metrics are in `workspace.airbnb_eda.gold_price_model_metrics`, a different table from the dashboard's `workspace.airbnb.gold_model_metrics`.
- **City C (week 2):** the same functions on the new city's table, without refitting:
```python
saved = pm.load_model(f"{MODEL_DIR}/{MODEL_NAME}.joblib")
c_data = load_source(["Santa_Clara_County"])
c_scored = pm.score_listings(c_data, saved["model"], saved["baseline"], saved["baseline_stay_type"], split_label="new_data")
pm.metrics_table(c_scored, {"baseline": "baseline_price", "model": "predicted_price"}, {"C": c_scored.price_status == "valid"})
write_rows(spark.createDataFrame(c_scored.astype(object).where(c_scored.notna(), None), schema=pm.GOLD_DDL), GOLD["predictions"],
           f"model_name = '{MODEL_NAME}' AND city IN ({sql_list(c_scored.city)})")      # adds the city C rows, nothing is overwritten
```
  Missing columns are filled with missing values (with a printed warning), unseen property types and neighbourhoods are tolerated; to change the model for city C, edit `FEATURES` or `CANDIDATES` in `src/price_model.py` and rerun this notebook.